Making Predictions with GlassPy
===============================

**Author:** Daniel R. Cassar



## Introduction



GlassPy provides three prediction models: `GlassNet`, `VITRIFY`, and `ViscNet`.

-   `GlassNet` is a multitask deep neural network that predicts 85 glass properties and the temperature dependence of viscosity.
-   `VITRIFY` is a classification model that estimates the probability of a given composition forming a glass.
-   `ViscNet` is a physics-informed deep neural network designed specifically to predict the temperature dependence of viscosity.

All three models were trained on data from the SciGlass database.



## GlassNet basic usage



Below is a minimal example of how to load and use the GlassNet model.



In [1]:
from glasspy.predict import GlassNet

model = GlassNet()
composition = "Li2O(SiO2)2"
predictions = model.predict(composition)

predictions

,T0,T1,T2,T3,T4,T5,T6,T7,T8,T9,...,Cp1673K,TMaxGrowthVelocity,MaxGrowthVelocity,CrystallizationPeak,CrystallizationOnset,SurfaceTensionAboveTg,SurfaceTension1173K,SurfaceTension1473K,SurfaceTension1573K,SurfaceTension1673K
0,1593.806265,1275.016879,1178.333278,1064.154863,956.53547,893.408138,857.634124,820.458542,779.195534,755.95825,...,1697.079401,1021.145482,-6.457833,921.905306,785.092585,0.309702,0.299307,0.321067,0.314809,0.318151


A composition can also be defined using a dictionary.



In [2]:
composition = {
    "SiO2": 2,
    "Li2O": 1,
}
predictions = model.predict(composition)

predictions

,T0,T1,T2,T3,T4,T5,T6,T7,T8,T9,...,Cp1673K,TMaxGrowthVelocity,MaxGrowthVelocity,CrystallizationPeak,CrystallizationOnset,SurfaceTensionAboveTg,SurfaceTension1173K,SurfaceTension1473K,SurfaceTension1573K,SurfaceTension1673K
0,1593.806265,1275.016879,1178.333278,1064.154863,956.53547,893.408138,857.634124,820.458542,779.195534,755.95825,...,1697.079401,1021.145482,-6.457833,921.905306,785.092585,0.309702,0.299307,0.321067,0.314809,0.318151


GlassNet also accepts `pandas` DataFrames as input. Note that each row represents a material and that only columns related to chemical composition can be present in the DataFrame.



In [3]:
import pandas as pd

data = [
    [1, 0, 2],
    [0, 1, 2],
    [1, 1, 2],
]

df = pd.DataFrame(data, columns=["Li2O", "Na2O", "SiO2"])
df

,Li2O,Na2O,SiO2
0,1,0,2
1,0,1,2
2,1,1,2


In [4]:
predictions = model.predict(df)

predictions

,T0,T1,T2,T3,T4,T5,T6,T7,T8,T9,...,Cp1673K,TMaxGrowthVelocity,MaxGrowthVelocity,CrystallizationPeak,CrystallizationOnset,SurfaceTensionAboveTg,SurfaceTension1173K,SurfaceTension1473K,SurfaceTension1573K,SurfaceTension1673K
0,1593.806528,1275.016816,1178.333278,1064.154806,956.535427,893.408194,857.634073,820.458542,779.195490,755.958250,...,1697.079401,1021.145445,-6.457834,921.905265,785.092630,0.309702,0.299307,0.321067,0.314809,0.318151
1,1681.328886,1513.411183,1279.021743,1123.006920,1035.724060,954.737548,912.160878,849.827548,809.466760,781.277537,...,1484.994346,1064.050420,-6.311865,958.442333,906.941741,0.273919,0.307567,0.289976,0.286678,0.271214
2,1436.034344,1233.614128,1101.165226,979.065643,921.910459,852.842167,785.283328,760.537597,724.843873,694.542212,...,1746.768638,1077.191654,-5.137043,864.736441,822.109318,0.293868,0.303260,0.318162,0.318062,0.293783


GlassNet can also predict viscosity and the MYEGA viscosity equation parameters.



In [5]:
predictions = model.predict_log10_viscosity(
    T=1000,
    composition=df,
)

predictions

array([3.44358113, 4.38299768, 2.72127358])

In [6]:
viscosity_parameters = model.viscosity_parameters(df)

viscosity_parameters

,log10_eta_infinity (Pa.s),Tg_MYEGA (K),fragility
0,-1.170492,697.863611,43.210780
1,-1.693836,710.538762,35.963149
2,-1.442656,642.028248,40.812665


## VITRIFY basic usage



Below is a minimal example of how to load and use the VITRIFY model. Like GlassNet, VITRIFY accepts strings, dictionaries, and `pandas` DataFrames as input. It returns `1` if the composition is predicted to form a glass under typical laboratory conditions, or `0` otherwise.



In [7]:
from glasspy.predict import VITRIFY

model = VITRIFY()
predictions = model.predict(df)

predictions

array([1, 1, 1], dtype=int64)

You can also retrieve the probability that the composition will form a glass, as estimated by the model.



In [8]:
probabilities = model.predict_proba_glass(df)

probabilities

array([0.97386932, 0.99601597, 0.78571427])

You can change the decision threshold for predicting that a composition forms a glass. The default value is 0.5 and this can be changed with the `thres` parameter. Increasing this threshold will make the model more strict on which compositions it predicts that forms a glass.



In [9]:
predictions = model.predict(df, thres=0.8)

predictions

array([1, 1, 0])

Two models are available, one trained on chemical composition (`CHEM`, the default option) and other trained on physicoquemical features (`FEATENG`). You can easily change the models with the `model` parameter. Both support the decision threshold tuning.



In [10]:
model = VITRIFY(model="FEATENG")
predictions = model.predict(df)

predictions

array([1, 1, 1], dtype=int64)

## ViscNet basic usage



ViscNet is used similarly to GlassNet. However, since GlassNet outperforms ViscNet on viscosity prediction, using GlassNet is generally recommended for that task. The minimal example below demonstrates how to load and use ViscNet.



In [11]:
from glasspy.predict import ViscNet

model = ViscNet()
log10_viscosity = model.predict(T=1000, composition=df)

log10_viscosity

array([5.1978884, 5.6581335, 6.0721507], dtype=float32)

In [12]:
fragility = model.predict_fragility(df)

print(fragility)

[34.436733 29.760376 34.16934 ]


In [13]:
Tg = model.predict_Tg(df)

print(Tg)

[748.3823  739.4691  788.51385]
